# Notebook 1 — Download, validate, merge and define the analytical sample

**Project:** BMI and HbA1c Analysis in US Adults (NHANES 2017–2018)
**Repository:** `nhanes-bmi-hba1c-analysis`
**Research question:** Is body mass index (BMI) associated with HbA1c levels in US adults?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<GITHUB_USERNAME>/nhanes-bmi-hba1c-analysis/blob/main/notebooks/01_download_validate_merge.ipynb)

*(Replace `<GITHUB_USERNAME>` in the badge link with your GitHub user name after you push the notebook.)*

---

## 1. Purpose and scope

This notebook builds the **data foundation** for the project. It does five things:

1. Downloads four public NHANES 2017–2018 files directly from the CDC and checks that each download is valid.
2. Reads them into pandas and checks their structure against the official CDC codebooks.
3. Merges them on the participant identifier `SEQN`, with the demographics file as the base.
4. Creates eligibility flags for the analytical sample, following the decisions below.
5. Produces a sample-flow table showing how many participants remain after each criterion.

**What this notebook deliberately does not do:** it fits no models, computes no weighted estimates and makes no statements about the US population. Survey weights and design variables are only *carried along* here so that later notebooks can use them correctly.

**Privacy and data handling:** no individual participant records are printed anywhere in this notebook. Only counts and summary tables are displayed. Raw and processed data stay in local working folders that are excluded from Git.

## 2. Analytical decisions implemented in this notebook

| Topic | Decision |
|---|---|
| Age | Participants aged **20 years or older** (`RIDAGEYR >= 20`; the CDC top-codes ages 80 and above as 80) |
| Examination | Examined in the Mobile Examination Center (MEC): `RIDSTATR = 2` |
| Pregnancy (`RIDEXPRG`) | `1` = confirmed pregnancy → **exclude**; `2` = not pregnant → include; `3` = could not be determined → **include** in the primary sample and **exclude** in a sensitivity sample; missing → **never** an exclusion reason by itself |
| Diabetes (`DIQ010`) | `1` = diagnosed diabetes; `2` = no diagnosed diabetes; `3` = borderline (kept as a **separate** category); `7` or `9` → set to missing |
| Missing data | Complete-case definition of the primary sample (non-missing BMI, HbA1c, age, sex, race/ethnicity) |
| Outliers | Not removed in this notebook (they are inspected in Notebook 2) |

## 3. Official data sources

All files come from the CDC National Center for Health Statistics (NCHS), NHANES 2017–2018 (cycle "J").

| File | Content | CDC documentation and codebook |
|---|---|---|
| `DEMO_J` | Demographics, sample weights, design variables | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.htm |
| `BMX_J` | Body measures (BMI) | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BMX_J.htm |
| `GHB_J` | Glycohemoglobin (HbA1c) | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/GHB_J.htm |
| `DIQ_J` | Diabetes questionnaire | https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DIQ_J.htm |

The "documented" counts and ranges used in the checks below were transcribed from these four codebook pages on **3 November 2026**. Where the notebook compares your downloaded data with them, a difference is reported rather than silently ignored.

## 4. How to run

* **Google Colab:** open the notebook, then choose *Runtime → Run all*. Nothing needs to be uploaded; everything is downloaded inside the session. Colab storage is temporary, so the downloaded and processed data disappear when the session ends (this is intended).
* **Locally:** run from the repository root or from the `notebooks/` folder with Python 3.10 or later.

## 5. Environment setup

**What the next cell does.** It installs `requests` and `pyarrow` only if they are missing, imports the libraries used in the notebook and prints their versions.

**Why print versions?** Recording the exact Python and library versions is part of reproducibility: if results ever differ between two runs, the versions are the first thing to compare.

**Why these libraries?**
* `pandas` / `numpy`: data handling.
* `requests`: downloading files over HTTPS, with explicit control over status codes and timeouts.
* `pyarrow`: lets pandas save the processed table in Parquet format (compact, and it preserves data types).

In [ ]:
import importlib.util
import subprocess
import sys

# Install only what is missing (Google Colab already ships pandas, numpy and requests).
_missing = [pkg for pkg in ("requests", "pyarrow") if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import hashlib
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 80)

RUN_STARTED_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
print(f"Python {platform.python_version()} | pandas {pd.__version__} | "
      f"numpy {np.__version__} | requests {requests.__version__}")
print(f"Run started (UTC): {RUN_STARTED_UTC}")

## 6. Project folders and Git safety

**What the next cells do.**

1. Detect whether the notebook runs in Google Colab or locally and choose the project root accordingly.
2. Create the folders `data/raw`, `data/processed` and `outputs/tables`.
3. Create (or complete) a `.gitignore` file that excludes all NHANES data files from Git.
4. If the project is already a Git repository, ask Git itself whether the data paths are ignored.

**Design choice.** Data live under `data/` and are *never* committed. Anyone who clones the repository regenerates the data by running this notebook, which makes the project reproducible without redistributing data. Small, aggregate outputs (sample-flow table, check log, download manifest) go to `outputs/tables/` and are safe to commit because they contain no participant-level information.

In [ ]:
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # A fresh, temporary project folder inside the Colab session.
    PROJECT_ROOT = Path("/content/nhanes-bmi-hba1c-analysis")
else:
    _cwd = Path.cwd()
    PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

for folder in (RAW_DIR, PROCESSED_DIR, TABLES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print(f"Running in Google Colab: {IN_COLAB}")
print(f"Project root:   {PROJECT_ROOT}")
print(f"Raw data:       {RAW_DIR}")
print(f"Processed data: {PROCESSED_DIR}")
print(f"Output tables:  {TABLES_DIR}")

In [ ]:
GITIGNORE_REQUIRED = [
    "# NHANES data: public, but deliberately never committed to this repository",
    "data/raw/*",
    "data/processed/*",
    "!data/raw/.gitkeep",
    "!data/processed/.gitkeep",
    "*.xpt",
    "*.XPT",
    "*.parquet",
    "# Python / Jupyter housekeeping",
    ".ipynb_checkpoints/",
    "__pycache__/",
    ".venv/",
    ".env",
]

gitignore_path = PROJECT_ROOT / ".gitignore"
existing = gitignore_path.read_text().splitlines() if gitignore_path.exists() else []
to_add = [line for line in GITIGNORE_REQUIRED if line not in existing]
if to_add:
    with open(gitignore_path, "a") as fh:
        if existing and existing[-1] != "":
            fh.write("\n")
        fh.write("\n".join(to_add) + "\n")
    print(f".gitignore updated ({len(to_add)} line(s) added): {gitignore_path}")
else:
    print(f".gitignore already contains all required rules: {gitignore_path}")

# .gitkeep files let Git keep the (otherwise empty) data folders in the repository.
for folder in (RAW_DIR, PROCESSED_DIR):
    (folder / ".gitkeep").touch()

# If this folder is already a Git repository, ask Git whether the data paths are ignored.
# `git check-ignore -q` exits with status 0 when the path IS ignored.
if (PROJECT_ROOT / ".git").exists():
    for rel_path in ("data/raw/DEMO_J.XPT", "data/processed/nhanes_2017_2018_analysis.parquet"):
        result = subprocess.run(["git", "check-ignore", "-q", rel_path], cwd=PROJECT_ROOT)
        print(f"Git ignores {rel_path}: {result.returncode == 0}")
        assert result.returncode == 0, f"{rel_path} is NOT ignored by Git; fix .gitignore before continuing."
else:
    print("Not a Git repository yet (normal in a fresh Colab session): Git check skipped.")

## 7. Configuration: files, URLs, expected variables and CDC reference values

**What this cell defines.**

* `CDC_URL_TEMPLATES`: the official CDC locations tried, in order, for each file. Two path styles are listed because CDC has used both for NHANES data files; the notebook records which one worked.
* `FILES`: for each file, the variables this project needs (`keep`), plus the documentation link.
* `CDC_REFERENCE`: row counts, value counts and value ranges **as documented in the CDC codebooks**. They are used only for validation, never to fill in or alter data.

**Why store reference values in the notebook?** A data pipeline should be able to detect when its inputs are not what the analysis plan assumed (for example, if CDC revised a file). Comparing against the published codebooks gives that protection.

**Severity of checks (used later).**
* *error*: a violation that would invalidate the pipeline, so the notebook stops (for example, duplicated `SEQN`).
* *warning*: a difference from the codebook that should be understood but does not by itself stop the pipeline.

In [ ]:
CDC_URL_TEMPLATES = [
    "https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/{name}.XPT",
    "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/{name}.xpt",
]
DOC_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/{name}.htm"

FILES = {
    "DEMO_J": {
        "description": "Demographics, sample weights and design variables",
        "keep": ["SEQN", "SDDSRVYR", "RIDSTATR", "RIDAGEYR", "RIAGENDR", "RIDRETH3",
                 "RIDEXPRG", "WTMEC2YR", "SDMVSTRA", "SDMVPSU"],
    },
    "BMX_J": {
        "description": "Body measures",
        "keep": ["SEQN", "BMXBMI", "BMDSTATS", "BMXWT", "BMXHT", "BMIWT"],
    },
    "GHB_J": {
        "description": "Glycohemoglobin (HbA1c)",
        "keep": ["SEQN", "LBXGH"],
    },
    "DIQ_J": {
        "description": "Diabetes questionnaire",
        "keep": ["SEQN", "DIQ010"],
    },
}

# Values transcribed from the CDC codebooks (see the documentation links above).
# "missing" refers to blank (.) values in the CDC frequency tables.
CDC_REFERENCE = {
    "DEMO_J": {
        "n_rows": 9254,
        "counts": {
            "SDDSRVYR": {10: 9254},
            "RIDSTATR": {1: 550, 2: 8704},
            "RIAGENDR": {1: 4557, 2: 4697},
            "RIDRETH3": {1: 1367, 2: 820, 3: 3150, 4: 2115, 6: 1168, 7: 634},
            "RIDEXPRG": {1: 55, 2: 966, 3: 89, "missing": 8144},
            "WTMEC2YR": {0: 550, "missing": 0},
        },
        "ranges": {"RIDAGEYR": (0, 80), "SDMVPSU": (1, 2), "SDMVSTRA": (134, 148)},
        "positive_ranges": {"WTMEC2YR": (2566.1838545, 419762.83649)},
        "continuous": ["WTMEC2YR"],
    },
    "BMX_J": {
        "n_rows": 8704,
        "counts": {
            "BMDSTATS": {1: 8019, 2: 295, 3: 283, 4: 107},
            "BMIWT": {1: 11, 3: 382, 4: 23, "missing": 8288},
            "BMXBMI": {"missing": 699},
        },
        "ranges": {"BMXBMI": (12.3, 86.2), "BMXWT": (3.2, 242.6), "BMXHT": (78.3, 197.7)},
        "positive_ranges": {},
        "continuous": ["BMXBMI"],
    },
    "GHB_J": {
        "n_rows": 6401,
        "counts": {"LBXGH": {"missing": 356}},
        "ranges": {"LBXGH": (3.8, 16.2)},
        "positive_ranges": {},
        "continuous": ["LBXGH"],
    },
    "DIQ_J": {
        "n_rows": 8897,
        "counts": {"DIQ010": {1: 893, 2: 7816, 3: 184, 7: 0, 9: 4}},
        "ranges": {},
        "positive_ranges": {},
        "continuous": [],
    },
}

# Design facts documented in DEMO_J: 15 masked strata, each with 2 masked PSUs (30 PSUs).
DOCUMENTED_N_STRATA = 15
DOCUMENTED_N_PSU = 30

for name, spec in FILES.items():
    print(f"{name}: {spec['description']} -> keep {len(spec['keep'])} variables | docs: {DOC_URL.format(name=name)}")

## 8. A small logging helper for data-quality checks

**What the next cell defines.** A function `check(...)` that (a) evaluates a condition, (b) prints `PASS`, `WARN` or `FAIL`, (c) stores the result in a list, and (d) stops the notebook if an *error*-level check fails.

**Why do this instead of scattered `assert` statements?** Every check ends up in one table (`CHECK_LOG`) that is saved at the end. That table is an auditable record of what was verified: exactly what you want to show an interviewer or a reviewer.

In [ ]:
CHECK_LOG = []

def check(section, description, passed, detail="", severity="error"):
    """Record a data-quality check.

    severity="error"   -> raise AssertionError if the check fails (the pipeline must stop).
    severity="warning" -> log a WARN and continue (needs human attention, not a stop).
    """
    passed = bool(passed)
    status = "PASS" if passed else ("FAIL" if severity == "error" else "WARN")
    CHECK_LOG.append({"section": section, "check": description, "status": status, "detail": detail})
    suffix = f" | {detail}" if detail else ""
    print(f"[{status}] {description}{suffix}")
    if status == "FAIL":
        raise AssertionError(f"Check failed: {description}{suffix}")

## 9. Download the official CDC files

**What the next cells do.** For each of the four files they try the official CDC URLs and verify the download before accepting it.

**Checks applied to every download**

| Check | Why it matters |
|---|---|
| HTTP status is 200 | Anything else means the server did not return the file |
| Content type is not HTML | Servers sometimes return an HTML error page with a 200 status |
| Byte count is above zero and equals `Content-Length` (when provided and the response is not compressed) | Detects empty or truncated downloads |
| File starts with the SAS Transport (XPORT) library header | Confirms the bytes really are an `.XPT` file |

**Other design choices**
* A `Retry` policy retries temporary server errors with a growing delay.
* Data are first written to a `.part` file and renamed only after validation, so a broken download can never be mistaken for a good one.
* A **SHA-256 fingerprint** of each file is stored in a manifest. If CDC silently revises a file, the fingerprint changes, so you can tell later which version of the data a result came from.
* If a valid file is already present in `data/raw`, it is reused (set `FORCE_DOWNLOAD = True` to download again).

In [ ]:
FORCE_DOWNLOAD = False
XPT_MAGIC = b"HEADER RECORD*******LIBRARY HEADER RECORD!!!!!!!"   # first bytes of any SAS XPORT file

def make_session():
    retry = Retry(total=3, backoff_factor=1.5,
                  status_forcelist=(429, 500, 502, 503, 504), allowed_methods=("GET",))
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update({"User-Agent": "nhanes-bmi-hba1c-analysis (portfolio project; python-requests)"})
    return session

def sha256_of(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def looks_like_xpt(path):
    with open(path, "rb") as fh:
        return fh.read(len(XPT_MAGIC)) == XPT_MAGIC

def download_xpt(name, session, force=False):
    """Download one NHANES file and return a manifest row. Raises RuntimeError if every URL fails."""
    destination = RAW_DIR / f"{name}.XPT"
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

    if destination.exists() and not force and destination.stat().st_size > 0 and looks_like_xpt(destination):
        return {"file": name, "url_used": "(existing local file reused)", "http_status": None,
                "content_type": None, "size_bytes": destination.stat().st_size,
                "sha256": sha256_of(destination), "retrieved_utc": now, "status": "reused"}

    attempts = []
    for template in CDC_URL_TEMPLATES:
        url = template.format(name=name)
        partial = destination.with_suffix(".part")
        try:
            with session.get(url, stream=True, timeout=(10, 120)) as response:
                status_code = response.status_code
                content_type = response.headers.get("Content-Type", "")
                if status_code != 200:
                    attempts.append(f"{url} -> HTTP {status_code}")
                    continue
                if "text/html" in content_type.lower():
                    attempts.append(f"{url} -> returned an HTML page instead of a data file")
                    continue
                n_bytes = 0
                with open(partial, "wb") as fh:
                    for chunk in response.iter_content(chunk_size=1 << 20):
                        fh.write(chunk)
                        n_bytes += len(chunk)
                declared = response.headers.get("Content-Length")
                compressed = response.headers.get("Content-Encoding")
        except requests.RequestException as exc:
            attempts.append(f"{url} -> {type(exc).__name__}")
            continue

        problems = []
        if n_bytes == 0:
            problems.append("empty file")
        if declared and not compressed and int(declared) != n_bytes:
            problems.append(f"size mismatch (declared {declared}, received {n_bytes})")
        if n_bytes > 0 and not looks_like_xpt(partial):
            problems.append("missing SAS XPORT header")
        if problems:
            partial.unlink(missing_ok=True)
            attempts.append(f"{url} -> rejected: {', '.join(problems)}")
            continue

        partial.replace(destination)
        return {"file": name, "url_used": url, "http_status": status_code, "content_type": content_type,
                "size_bytes": n_bytes, "sha256": sha256_of(destination), "retrieved_utc": now,
                "status": "downloaded"}

    raise RuntimeError(
        f"Could not download {name}. Attempts:\n  " + "\n  ".join(attempts) +
        f"\nOpen the CDC page for this file ({DOC_URL.format(name=name)}) and check the data link, "
        "or update CDC_URL_TEMPLATES."
    )

In [ ]:
session = make_session()
manifest_rows = [download_xpt(name, session, force=FORCE_DOWNLOAD) for name in FILES]
manifest = pd.DataFrame(manifest_rows)

# Display a shortened fingerprint; the full SHA-256 is saved in the CSV later.
display(manifest.assign(sha256=manifest["sha256"].str[:12] + "...")[
    ["file", "status", "http_status", "size_bytes", "sha256", "url_used"]])

for row in manifest_rows:
    path = RAW_DIR / f"{row['file']}.XPT"
    check("download", f"{row['file']}: file exists, is non-empty and has a valid XPORT header",
          path.exists() and path.stat().st_size > 0 and looks_like_xpt(path),
          detail=f"{row['size_bytes']:,} bytes ({row['status']})")

## 10. Read the SAS Transport (`.XPT`) files into pandas

**What the next cells do.** `pandas.read_sas(..., format="xport")` reads each file. The notebook then repairs one known quirk of this reader and prints only the *shape* (rows × columns) of each table.

**The zero-value quirk, explained.** SAS Transport files store numbers in IBM floating-point format. pandas converts them to ordinary floating-point numbers, but a true `0` is converted to a tiny non-zero number (about `5.4e-79`) instead of exactly `0.0`.

This matters for our data: the examination weight `WTMEC2YR` is exactly 0 for people who were interviewed but not examined, and later logic (and the checks below) compares values with `== 0`. The helper therefore sets any number with an absolute value below `1e-30` to exactly 0. No real NHANES measurement is anywhere near that small, so no genuine data value can be affected. The number of corrected cells is reported for transparency.

**Privacy.** Only shapes and column names are printed; no rows are shown.

In [ ]:
def read_xpt(path):
    """Read an XPORT file; repair the IBM-float zero artefact; return (DataFrame, n_cells_repaired)."""
    df = pd.read_sas(path, format="xport")
    numeric_cols = df.select_dtypes(include="number").columns
    near_zero = df[numeric_cols].abs() < 1e-30          # NaN compares as False, so missing values are untouched
    n_repaired = int(near_zero.to_numpy().sum())
    df[numeric_cols] = df[numeric_cols].mask(near_zero, 0.0)
    return df, n_repaired

raw = {}
read_summary = []
for name in FILES:
    df, n_repaired = read_xpt(RAW_DIR / f"{name}.XPT")
    raw[name] = df
    read_summary.append({"file": name, "rows": len(df), "columns": df.shape[1],
                         "zero_artefact_cells_repaired": n_repaired})

display(pd.DataFrame(read_summary))

## 11. Structural validation: variables, data types, `SEQN` and row counts

**Checks in the next cell**

1. **Expected variables exist** in each file (a missing variable would break everything downstream).
2. **Expected variables are numeric.** SAS Transport stores these as numbers; the codes (for example `1 = Male`) are numeric codes, not text.
3. **`SEQN` is valid and unique** in every file. `SEQN` is the NHANES respondent sequence number: the key used to link files. If it were duplicated, a merge would silently multiply rows.
4. **Row counts equal the CDC codebooks** (error-level: a different row count means the input is not the file described in the analysis plan).

After these checks `SEQN` is converted from a floating-point number to an integer.

In [ ]:
for name, spec in FILES.items():
    df = raw[name]

    missing_vars = [v for v in spec["keep"] if v not in df.columns]
    check("structure", f"{name}: all {len(spec['keep'])} expected variables are present",
          not missing_vars, detail=f"missing: {missing_vars}" if missing_vars else "")

    non_numeric = [v for v in spec["keep"] if not pd.api.types.is_numeric_dtype(df[v])]
    check("structure", f"{name}: expected variables are numeric", not non_numeric,
          detail=f"non-numeric: {non_numeric}" if non_numeric else "")

    seqn = df["SEQN"]
    check("structure", f"{name}: SEQN has no missing values", seqn.notna().all())
    check("structure", f"{name}: SEQN values are whole numbers", (seqn.dropna() % 1 == 0).all())
    n_duplicated = int(seqn.duplicated().sum())
    check("structure", f"{name}: SEQN is unique", n_duplicated == 0,
          detail=f"{n_duplicated} duplicated SEQN value(s)" if n_duplicated else "")
    df["SEQN"] = seqn.astype("int64")

    expected_rows = CDC_REFERENCE[name]["n_rows"]
    check("structure", f"{name}: row count equals the CDC codebook ({expected_rows:,})",
          len(df) == expected_rows, detail=f"observed {len(df):,}")

## 12. Verifying documented codes, ranges and frequencies

**Why this section matters.** The analysis plan relies on specific numeric codes, for example `RIDEXPRG = 3` ("cannot ascertain"), `DIQ010 = 7/9` (refused / don't know) and `WTMEC2YR = 0` (not examined). Before using them, the notebook confirms that:

* every value in each *categorical* variable is one of the **documented codes** (no surprise codes; continuous variables such as BMI, HbA1c and the weight are checked by range and by their count of blanks instead);
* the **frequency of each code** matches the CDC frequency table;
* continuous variables fall inside the **documented range**;
* the design variables show the documented structure (**15 strata × 2 PSUs = 30 PSUs**), which later determines the design degrees of freedom.

**Severity.** These checks are *warnings*, not errors, because small differences can have benign explanations. Any WARN should still be read and understood before the analysis proceeds.

In [ ]:
def compare_counts(file_name, df, variable, documented):
    """Compare observed code frequencies with the CDC codebook frequency table."""
    series = df[variable]
    rows = []
    for code_value, documented_n in documented.items():
        observed_n = int(series.isna().sum()) if code_value == "missing" else int((series == code_value).sum())
        rows.append({"file": file_name, "variable": variable, "code": str(code_value),
                     "cdc_codebook_n": documented_n, "observed_n": observed_n,
                     "match": observed_n == documented_n})
    documented_codes = [c for c in documented if c != "missing"]
    unexpected = sorted(set(series.dropna().unique()) - set(documented_codes))
    return pd.DataFrame(rows), unexpected

def in_range(series, low, high, positive_only=False, tolerance=1e-3):
    values = series.dropna()
    if positive_only:
        values = values[values > 0]
    if values.empty:
        return False, "no values"
    ok = (values.min() >= low - tolerance) and (values.max() <= high + tolerance)
    return ok, f"observed {values.min():g} to {values.max():g}; documented {low:g} to {high:g}"

count_tables = []
for name, ref in CDC_REFERENCE.items():
    df = raw[name]
    for variable, documented in ref["counts"].items():
        table, unexpected = compare_counts(name, df, variable, documented)
        count_tables.append(table)
        n_mismatch = int((~table["match"]).sum())
        if variable not in ref["continuous"]:      # continuous variables have no "code list" to test
            check("codes", f"{name}.{variable}: no undocumented codes", not unexpected,
                  detail=f"unexpected codes (first 10): {unexpected[:10]}" if unexpected else "", severity="warning")
        check("codes", f"{name}.{variable}: code frequencies match the CDC codebook", n_mismatch == 0,
              detail=f"{n_mismatch} of {len(table)} code(s) differ" if n_mismatch else "", severity="warning")
    for variable, (low, high) in ref["ranges"].items():
        ok, detail = in_range(df[variable], low, high)
        check("ranges", f"{name}.{variable}: values inside the documented range", ok, detail=detail, severity="warning")
    for variable, (low, high) in ref["positive_ranges"].items():
        ok, detail = in_range(df[variable], low, high, positive_only=True)
        check("ranges", f"{name}.{variable}: positive values inside the documented range", ok, detail=detail, severity="warning")

# Design structure: the degrees of freedom of later survey analyses depend on it.
demo_raw = raw["DEMO_J"]
n_strata = demo_raw["SDMVSTRA"].nunique()
n_psu = demo_raw[["SDMVSTRA", "SDMVPSU"]].drop_duplicates().shape[0]
check("design", f"DEMO_J: {DOCUMENTED_N_STRATA} strata and {DOCUMENTED_N_PSU} PSUs (strata x PSU combinations)",
      n_strata == DOCUMENTED_N_STRATA and n_psu == DOCUMENTED_N_PSU,
      detail=f"observed {n_strata} strata and {n_psu} PSUs")

code_comparison = pd.concat(count_tables, ignore_index=True)
display(code_comparison)

## 13. Merge the four files on `SEQN`

**What the next cell does.** It starts from `DEMO_J` (the *base* table) and attaches the variables from `BMX_J`, `GHB_J` and `DIQ_J` with **left joins** on `SEQN`.

**Why `DEMO_J` is the base.** It holds the survey weight and design variables (`WTMEC2YR`, `SDMVSTRA`, `SDMVPSU`) for *every* sampled participant. Keeping all of them is essential later: the NHANES design must be defined on the full sample, and the analytical sample is then defined as a *subpopulation* inside it. A merge that started from a smaller file would silently drop participants and break that logic.

**Safeguards**
* `validate="one_to_one"` makes pandas raise an error if `SEQN` is duplicated on either side.
* Before each join, the notebook checks that no `SEQN` exists in the other file but not in `DEMO_J` (such rows would be silently lost by a left join).
* After each join, the notebook confirms that the row count is unchanged.
* An indicator column `in_<file>` records whether each participant had a row in that file. Participants without a row get missing values, which is expected (for example, people who were not examined have no body measures).

**Plausibility of missing rows (warnings).** Everyone without a `BMX_J` row should be a participant who was not examined, and everyone without a `DIQ_J` row should be under 1 year old (the DIQ_J codebook says participants aged 1 year and older were eligible).

In [ ]:
base = raw["DEMO_J"][FILES["DEMO_J"]["keep"]].copy()
n_base = len(base)
merged = base.copy()
merge_report = []

for name in ("BMX_J", "GHB_J", "DIQ_J"):
    right = raw[name][FILES[name]["keep"]].copy()

    n_orphans = int((~right["SEQN"].isin(base["SEQN"])).sum())
    check("merge", f"{name}: every SEQN also exists in DEMO_J (no rows would be lost)", n_orphans == 0,
          detail=f"{n_orphans} orphan row(s)" if n_orphans else "")

    n_before = len(merged)
    merged = merged.merge(right, on="SEQN", how="left", validate="one_to_one")
    check("merge", f"After joining {name}: row count unchanged ({n_before:,})", len(merged) == n_before,
          detail=f"observed {len(merged):,}")

    merged[f"in_{name}"] = merged["SEQN"].isin(right["SEQN"])
    merge_report.append({"file": name, "rows_in_file": len(right),
                         "matched_to_DEMO_J": int(merged[f"in_{name}"].sum()),
                         "DEMO_J_without_row_in_file": int((~merged[f"in_{name}"]).sum())})

check("merge", f"Final merged table keeps all {n_base:,} DEMO_J participants", len(merged) == n_base)
check("merge", "SEQN is unique in the merged table", merged["SEQN"].is_unique)
check("merge", "No duplicated column names after merging", not merged.columns.duplicated().any())

no_bmx = ~merged["in_BMX_J"]
check("merge", "Participants without a BMX_J row are all non-examined (RIDSTATR = 1)",
      (merged.loc[no_bmx, "RIDSTATR"] == 1).all(),
      detail=f"{int((merged.loc[no_bmx, 'RIDSTATR'] != 1).sum())} examined participant(s) without body measures",
      severity="warning")
no_diq = ~merged["in_DIQ_J"]
check("merge", "Participants without a DIQ_J row are all under 1 year old (RIDAGEYR = 0)",
      (merged.loc[no_diq, "RIDAGEYR"] == 0).all(),
      detail=f"{int((merged.loc[no_diq, 'RIDAGEYR'] != 0).sum())} participant(s) aged 1+ without a DIQ_J row",
      severity="warning")

display(pd.DataFrame(merge_report))
print(f"Merged table: {merged.shape[0]:,} rows x {merged.shape[1]} columns")

## 14. Recode special codes and derive the diabetes category

**Special codes.** In the CDC's `DIQ010` coding, `7` means *refused* and `9` means *don't know*. These are not answers about diabetes, so they are converted to missing (`NaN`). The codebook for `DIQ_J` was checked: `DIQ010` has the codes 1 (Yes), 2 (No), 3 (Borderline), 7 (Refused) and 9 (Don't know). The checks in section 12 confirmed that no other codes occur in your downloaded file.

**Diabetes category (`diabetes_cat`).** Created from `DIQ010` according to the project decision:

| `DIQ010` | `diabetes_cat` |
|---|---|
| 1 | Diagnosed diabetes |
| 2 | No diagnosed diabetes |
| 3 | Borderline diabetes (**kept separate**, not merged with "No diagnosed diabetes") |
| 7, 9, or no `DIQ_J` row | Missing |

**What is *not* recoded here.** `BMXBMI` and `LBXGH` have no special codes (their missing values are already blank). `RIDEXPRG` is handled in the next section because its code `3` is a real category and its blanks have a structural meaning.

In [ ]:
n_special = int(merged["DIQ010"].isin([7, 9]).sum())
merged["DIQ010"] = merged["DIQ010"].mask(merged["DIQ010"].isin([7, 9]))
print(f"DIQ010: {n_special} value(s) coded 7 (refused) or 9 (don't know) set to missing")

DIABETES_LABELS = {1: "Diagnosed diabetes", 2: "No diagnosed diabetes", 3: "Borderline diabetes"}
merged["diabetes_cat"] = pd.Categorical(merged["DIQ010"].map(DIABETES_LABELS),
                                        categories=list(DIABETES_LABELS.values()))

check("recode", "DIQ010 now contains only the codes 1, 2, 3 or missing",
      set(merged["DIQ010"].dropna().unique()) <= {1, 2, 3})
display(merged["diabetes_cat"].value_counts(dropna=False).rename_axis("diabetes_cat").to_frame("n_all_participants"))

## 15. Pregnancy status (`RIDEXPRG`): what is missing, and why

The CDC documents that `RIDEXPRG` is released **only for females aged 20–44** at the time of the examination. Blank values therefore mean different things:

| Group | `RIDEXPRG` | Meaning |
|---|---|---|
| Women 20–44 | 1 | Pregnant |
| Women 20–44 | 2 | Not pregnant |
| Women 20–44 | 3 | Pregnancy status could not be determined |
| Men, and women outside 20–44 | blank | **Not applicable / not released**, which is *not* evidence of pregnancy |

**The rule used in this project**

* `RIDEXPRG = 1` → excluded;
* `RIDEXPRG = 2` → included;
* `RIDEXPRG = 3` → included in the primary sample, excluded in the sensitivity sample;
* blank → **never excluded** because of this variable.

**Checks below** (warnings, because they describe the data rather than protect the pipeline): that non-missing values really occur only in women aged 20–44, and how many examined women aged 20–44 have a blank value. A cross-tabulation for examined adults shows the structure in numbers.

**Known limitation (from the CDC documentation).** Pregnancy status is not released for women aged 45 and older, so a very small number of pregnancies in that group cannot be identified from the public file. The CDC's BMX_J notes also say that body measures for some pregnant women outside the released age range are not reported, and such participants will drop out through missing BMI.

In [ ]:
is_woman_20_44 = (merged["RIAGENDR"] == 2) & merged["RIDAGEYR"].between(20, 44)
has_status = merged["RIDEXPRG"].notna()

n_outside_target = int((has_status & ~is_woman_20_44).sum())
check("pregnancy", "RIDEXPRG is non-missing only for women aged 20-44", n_outside_target == 0,
      detail=f"{n_outside_target} participant(s) with a value outside the target group" if n_outside_target else "",
      severity="warning")

examined_women_20_44 = is_woman_20_44 & (merged["RIDSTATR"] == 2)
n_blank_in_target = int((examined_women_20_44 & ~has_status).sum())
check("pregnancy", "Every examined woman aged 20-44 has a RIDEXPRG value", n_blank_in_target == 0,
      detail=f"{n_blank_in_target} examined woman/women aged 20-44 with a blank value" if n_blank_in_target else "",
      severity="warning")

adults_examined = merged[(merged["RIDSTATR"] == 2) & (merged["RIDAGEYR"] >= 20)]
group = np.select(
    [(adults_examined["RIAGENDR"] == 2) & adults_examined["RIDAGEYR"].between(20, 44),
     adults_examined["RIAGENDR"] == 2,
     adults_examined["RIAGENDR"] == 1],
    ["Women aged 20-44", "Women aged 45+", "Men aged 20+"], default="Other")
status = adults_examined["RIDEXPRG"].map({1: "1 Pregnant", 2: "2 Not pregnant", 3: "3 Cannot ascertain"})
status = status.astype("object").where(status.notna(), "Blank (not released)")
display(pd.crosstab(pd.Series(group, index=adults_examined.index, name="Examined adults"),
                    status.rename("RIDEXPRG status")))

## 16. Eligibility flags

**Design principle: flag, do not delete.** The notebook adds Boolean (`True`/`False`) columns instead of removing rows. All 9,254-row structure of `DEMO_J` stays intact, which is what the survey design needs (see section 13), and every exclusion remains visible and auditable.

**Building blocks**

| Flag | Rule |
|---|---|
| `flag_examined` | `RIDSTATR == 2` |
| `flag_age_20plus` | `RIDAGEYR >= 20` |
| `flag_preg_confirmed` | `RIDEXPRG == 1` |
| `flag_preg_indeterminate` | `RIDEXPRG == 3` |
| `flag_not_preg_confirmed` | the opposite of `flag_preg_confirmed`: **True for blanks**, which is how "blank never excludes" is implemented |
| `flag_has_bmi`, `flag_has_hba1c` | `BMXBMI`, `LBXGH` not missing |
| `flag_has_covariates` | age, sex and race/ethnicity (`RIDRETH3`) not missing |

**Sample definitions**

* `in_eligible_adults` = examined **and** aged 20+ **and** not a confirmed pregnancy.
* `in_primary_sample` = eligible adults **and** non-missing BMI, HbA1c, age, sex and race/ethnicity (complete cases for the primary model).
* `in_sensitivity_excl_preg_indeterminate` = primary sample **excluding** `RIDEXPRG == 3`. Only the *flag* is created here; the sensitivity analysis itself comes later.

**Implementation note on missing values.** In pandas, `NaN == 1` evaluates to `False`. Therefore `merged["RIDEXPRG"].eq(1)` is `False` for blanks, and the flag for "not a confirmed pregnancy" is automatically `True` for them.

**Consistency checks.** The examination weight must be positive exactly when the participant was examined, and body measures and HbA1c should exist only for examined participants.

In [ ]:
merged["flag_examined"] = merged["RIDSTATR"].eq(2)
merged["flag_age_20plus"] = merged["RIDAGEYR"].ge(20)
merged["flag_preg_confirmed"] = merged["RIDEXPRG"].eq(1)
merged["flag_preg_indeterminate"] = merged["RIDEXPRG"].eq(3)
merged["flag_not_preg_confirmed"] = ~merged["flag_preg_confirmed"]      # True for 2, 3 and blanks
merged["flag_has_bmi"] = merged["BMXBMI"].notna()
merged["flag_has_hba1c"] = merged["LBXGH"].notna()
merged["flag_has_covariates"] = merged[["RIDAGEYR", "RIAGENDR", "RIDRETH3"]].notna().all(axis=1)

merged["in_eligible_adults"] = (merged["flag_examined"] & merged["flag_age_20plus"]
                                & merged["flag_not_preg_confirmed"])
merged["in_primary_sample"] = (merged["in_eligible_adults"] & merged["flag_has_bmi"]
                               & merged["flag_has_hba1c"] & merged["flag_has_covariates"])
merged["in_sensitivity_excl_preg_indeterminate"] = (merged["in_primary_sample"]
                                                    & ~merged["flag_preg_indeterminate"])

check("flags", "Examination weight is positive exactly when RIDSTATR = 2",
      ((merged["WTMEC2YR"] > 0) == merged["flag_examined"]).all())
check("flags", "BMI is present only for examined participants", (~merged["flag_has_bmi"] | merged["flag_examined"]).all())
check("flags", "HbA1c is present only for examined participants", (~merged["flag_has_hba1c"] | merged["flag_examined"]).all())
check("flags", "Primary sample is a subset of the eligible adults", (~merged["in_primary_sample"] | merged["in_eligible_adults"]).all())
check("flags", "Sensitivity sample is a subset of the primary sample",
      (~merged["in_sensitivity_excl_preg_indeterminate"] | merged["in_primary_sample"]).all())
check("flags", "Every participant in the primary sample has a positive weight and design variables",
      (merged.loc[merged["in_primary_sample"], "WTMEC2YR"] > 0).all()
      and merged.loc[merged["in_primary_sample"], ["SDMVSTRA", "SDMVPSU"]].notna().all().all())

flag_columns = [c for c in merged.columns if c.startswith(("flag_", "in_"))]
display(merged[flag_columns].sum().astype(int).rename("n_True").to_frame())

## 17. Sample-flow table

**What it shows.** The number of participants remaining after each criterion applied **in sequence**, and how many were removed at each step. The order of the criteria affects the number removed at each *step* (not the final sample), so the order used is stated explicitly.

**Reading the table.**
* Steps 1–3 define the *eligible adults* (`in_eligible_adults`).
* Steps 4–6 apply the complete-case requirements and give the *primary sample* (`in_primary_sample`).
* These are **unweighted sample counts** of participants. They say nothing about the size of any population, because no survey weights are used here.

The notebook checks that the last row of the table equals the primary-sample flag and that counts never increase from step to step.

In [ ]:
steps = [
    ("0", "All participants in DEMO_J", pd.Series(True, index=merged.index)),
    ("1", "Examined in the MEC (RIDSTATR = 2)", merged["flag_examined"]),
    ("2", "Aged 20 years or older (RIDAGEYR >= 20)", merged["flag_age_20plus"]),
    ("3", "Not a confirmed pregnancy (RIDEXPRG != 1; blanks are kept)", merged["flag_not_preg_confirmed"]),
    ("4", "Non-missing BMI (BMXBMI)", merged["flag_has_bmi"]),
    ("5", "Non-missing HbA1c (LBXGH)", merged["flag_has_hba1c"]),
    ("6", "Non-missing age, sex and race/ethnicity", merged["flag_has_covariates"]),
]

remaining = pd.Series(True, index=merged.index)
flow_rows, previous_n = [], None
for step, criterion, mask in steps:
    remaining = remaining & mask
    n_remaining = int(remaining.sum())
    flow_rows.append({"step": step, "criterion": criterion, "n_remaining": n_remaining,
                      "n_excluded_at_step": 0 if previous_n is None else previous_n - n_remaining})
    previous_n = n_remaining

sample_flow = pd.DataFrame(flow_rows)
sample_flow["pct_of_all_participants"] = (100 * sample_flow["n_remaining"] / n_base).round(1)

check("sample_flow", "Counts never increase from one step to the next", sample_flow["n_remaining"].is_monotonic_decreasing)
check("sample_flow", "Step 3 equals the eligible-adults flag", sample_flow.loc[3, "n_remaining"] == merged["in_eligible_adults"].sum())
check("sample_flow", "Final step equals the primary-sample flag", sample_flow.iloc[-1]["n_remaining"] == merged["in_primary_sample"].sum())
check("sample_flow", "Step 1 equals the number of examined participants", sample_flow.loc[1, "n_remaining"] == (merged["RIDSTATR"] == 2).sum())

display(sample_flow)

## 18. Supplementary summaries

These aggregate tables prepare the ground for Notebook 2. They contain counts only.

1. **Which eligible adults are missing BMI and/or HbA1c?** Missingness is the main reason participants leave the sample after step 3.
2. **Pregnancy-related counts** among examined adults aged 20+ (confirmed pregnancy, indeterminate status) and the size of the sensitivity sample compared with the primary sample.
3. **Diabetes category** within the primary sample. A blank category means refused, don't know, or no `DIQ_J` answer.

In [ ]:
eligible = merged[merged["in_eligible_adults"]]
missing_pattern = pd.crosstab(
    eligible["flag_has_bmi"].map({True: "BMI present", False: "BMI missing"}),
    eligible["flag_has_hba1c"].map({True: "HbA1c present", False: "HbA1c missing"}),
    margins=True, margins_name="Total")
print("Eligible adults: availability of BMI and HbA1c")
display(missing_pattern)

n_primary = int(merged["in_primary_sample"].sum())
n_sens = int(merged["in_sensitivity_excl_preg_indeterminate"].sum())
pregnancy_summary = pd.DataFrame({
    "quantity": [
        "Examined adults aged 20+ with a confirmed pregnancy (RIDEXPRG = 1), excluded",
        "Examined adults aged 20+ with indeterminate status (RIDEXPRG = 3)",
        "Primary sample (n)",
        "Sensitivity sample excluding RIDEXPRG = 3 (n)",
        "Participants removed in the sensitivity sample",
    ],
    "n": [
        int((merged["flag_examined"] & merged["flag_age_20plus"] & merged["flag_preg_confirmed"]).sum()),
        int((merged["flag_examined"] & merged["flag_age_20plus"] & merged["flag_preg_indeterminate"]).sum()),
        n_primary, n_sens, n_primary - n_sens,
    ],
})
print("Pregnancy handling")
display(pregnancy_summary)

primary = merged[merged["in_primary_sample"]]
diabetes_in_primary = (primary["diabetes_cat"].astype("object").fillna("Missing (refused / don't know / not asked)")
                       .value_counts().rename_axis("diabetes_cat").to_frame("n_primary_sample"))
print("Primary sample: diabetes category")
display(diabetes_in_primary)

## 19. Save the processed dataset and the aggregate tables

**What is saved**

| Item | Location | Committed to GitHub? |
|---|---|---|
| Merged table with flags (one row per participant) | `data/processed/nhanes_2017_2018_analysis.parquet` | **No** (excluded by `.gitignore`) |
| Raw `.XPT` files | `data/raw/` | **No** (excluded by `.gitignore`) |
| Sample-flow table | `outputs/tables/01_sample_flow.csv` | Yes (aggregate only) |
| Download manifest (URL, size, SHA-256, time) | `outputs/tables/01_download_manifest.csv` | Yes (no participant data) |
| Data-quality check log | `outputs/tables/01_check_log.csv` | Yes (no participant data) |

**Why Parquet?** It is compact, fast and preserves data types (integers, booleans, categories), unlike CSV, which turns everything into text.

After writing, the notebook reads the Parquet file back and verifies that it has the same shape and the same flag totals. This catches silent write errors.

In [ ]:
PROCESSED_FILE = PROCESSED_DIR / "nhanes_2017_2018_analysis.parquet"
merged.to_parquet(PROCESSED_FILE, index=False)

reloaded = pd.read_parquet(PROCESSED_FILE)
check("save", "Re-loaded processed file has the same shape", reloaded.shape == merged.shape,
      detail=f"{reloaded.shape[0]:,} rows x {reloaded.shape[1]} columns")
check("save", "Re-loaded flag totals are identical",
      reloaded[flag_columns].sum().equals(merged[flag_columns].sum()))
del reloaded

sample_flow.to_csv(TABLES_DIR / "01_sample_flow.csv", index=False)
manifest.to_csv(TABLES_DIR / "01_download_manifest.csv", index=False)

print("Files written (paths and sizes only):")
for path in (PROCESSED_FILE, TABLES_DIR / "01_sample_flow.csv", TABLES_DIR / "01_download_manifest.csv"):
    print(f"  {path.relative_to(PROJECT_ROOT)}  ({path.stat().st_size:,} bytes)")

## 20. Summary of all checks

The table below counts how many checks passed, produced warnings or failed (a failure would already have stopped the notebook). **Every WARN should be read.** A warning means the data differ from the CDC documentation or from an expectation in some way that deserves an explanation in the project's decision log before analysis continues.

In [ ]:
check_log = pd.DataFrame(CHECK_LOG)
check_log.to_csv(TABLES_DIR / "01_check_log.csv", index=False)

print(check_log["status"].value_counts().rename_axis("status").to_frame("n_checks").to_string())
warnings_table = check_log[check_log["status"] == "WARN"]
if warnings_table.empty:
    print("\nNo warnings: all documented counts, ranges and codes match the CDC codebooks.")
else:
    print("\nChecks that raised a warning (review these):")
    display(warnings_table[["section", "check", "detail"]])
print(f"\nSaved: {(TABLES_DIR / '01_check_log.csv').relative_to(PROJECT_ROOT)}")

## 21. Assumptions, limitations and what comes next

**Assumptions made in this notebook**

1. **Download locations.** The two CDC URL patterns in the configuration cell are the ones the notebook tries. The manifest records which URL actually worked.
2. **XPORT reading.** `pandas.read_sas` is used for `.XPT` files, with a documented repair of the near-zero artefact (section 10).
3. **Reference counts.** CDC codebook counts were transcribed on 3 November 2026. If CDC revises a file, row-count checks (errors) and code-frequency checks (warnings) will flag it, and the SHA-256 in the manifest identifies the file version.
4. **Pregnancy.** Blank `RIDEXPRG` outside women aged 20–44 is treated as "not applicable"; pregnancies in women aged 45+ cannot be identified from the public file.
5. **Diabetes.** `DIQ010` is self-reported ("a doctor told you"), so undiagnosed diabetes cannot be identified from this variable.
6. **Age.** `RIDAGEYR` is top-coded at 80.
7. **Complete cases.** The primary sample requires non-missing BMI and HbA1c. Whether that introduces bias is examined in Notebook 2 (included versus excluded participants).

**What this notebook does not establish**

* Nothing here is representative of the US population. Counts are unweighted participant counts.
* No outlier handling or distributions yet (Notebook 2).
* No survey-design object, no `svy` or R validation, no models (Notebook 3 and the validation script).

**Next step.** Notebook 2 will load `data/processed/nhanes_2017_2018_analysis.parquet` (or simply re-run this notebook in a new Colab session) and cover cleaning, missingness, outlier inspection and exploratory analysis.